# 05. CIFAR-100, capacity and accuracy after every deletion (M5)

What happens to accuracy when the method runs out of available directions?

Notebook 04 saved accuracy at 5, 10, 20, and 30 deletions. This notebook measures every
step from 0 through 30, so we can inspect the transition around deletion 16. It studies
only the paper's in-training method, at lambda 2.0, using three seeds.

**Decision rule fixed before execution:** test whether free rank follows the full-rank
prediction and reaches zero at 64 / 4 = 16. After measured exhaustion, test whether new
updates and changes to the head stay below the numerical tolerances in `config.yaml`.
Report accuracy curves even if either check fails. These checks concern the reconstruction,
not proof of what the authors ran. Compare Fig. 3's stated retain level and remaining rank
as references, not matched baselines. No automatic claim of successful replication follows.

**Limits:** the public checkpoint differs from the paper's checkpoint. Training settings and
class order are our documented choices. Accuracy on forgotten classes measures classification
suppression, not verified removal of training influence. All seeds share one frozen backbone.

**Resources:** select a CPU runtime with standard RAM. No GPU is needed. Feature extraction
runs once, followed by 90 small adapter fits. Runtime has not been measured for this notebook.
Feature tensors require about 15 MB in float32, or 31 MB in float64, before labels and overhead.
Allow additional disk space for CIFAR-100, its archive, checkpoint, and dependencies; total
peak storage is not measured. Small results go to R2; feature caches remain local.

## Words used in this notebook

- **Backbone:** the frozen image network. It turns each image into 64 feature values.
- **Feature cache:** those saved values for 50,000 training and 10,000 test images.
- **Head:** the final layer, mapping 64 features to 100 class scores with a 64 by 100 weight matrix.
- **Adapter:** a learned correction formed from matrices of size 64 by 4 and 4 by 100.
- **Rank:** the number of independent directions used by a matrix. The configured adapter rank is four.
- **Projection / projector:** the matrix that restricts each new adapter to unused input directions.
- **Free rank:** the number of directions still available. It starts at 64.
- **Budget / exhaustion:** the predicted number of full-rank deletions, 64 / 4 = 16, and the point
  when no directions remain. Actual exhaustion is measured separately for each seed.
- **Lambda:** the multiplier used when subtracting adapters. Here it is 2.0; it does not enter training.
- **Retain accuracy:** percentage correct on classes not yet requested for deletion.
- **Forget-all accuracy:** percentage correct over all classes requested so far.
- **Forget-current accuracy:** percentage correct on the newest requested class only.
- **Full accuracy:** percentage correct over the unchanged full test set.
- **Seed:** the number controlling random initialization and batch order. We use 0, 1, and 2.
- **Standard deviation:** spread across seeds. Shading shows this spread, not a confidence interval.
- **Weight drift:** largest absolute weight difference from the head at measured exhaustion.
- **Float64:** double-precision arithmetic used for training and rank accounting.
- **Run label / provenance:** the unique result name and the settings, code, and input identities behind it.
- **JSONL / CSV:** text formats for individual progress records and tabular results.
- **R2:** persistent object storage. Its credentials are read from the three DSR secrets.
- **Resume boundary:** a completed seed. An interrupted seed restarts from its first deletion.

## 0. Setup on Colab

Clone and install the project from the named main branch. Replace REVISION with a commit
hash to pin it. Setup refuses to proceed after the project has been imported: restart the
runtime first. Existing local checkouts are used as they are. Pip installs missing or
incompatible dependencies; already satisfied dependencies are retained.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sumitasthana/DSR.git"
REVISION = "main"   # Replace with a commit hash to pin this run.

if any(name == "src" or name.startswith("src.") for name in sys.modules):
    raise RuntimeError("The experiment package is already imported. "
                       "Restart the runtime before running setup, or you will "
                       "silently test stale code.")

if "google.colab" in sys.modules:
    if not Path("DSR").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    subprocess.run(["git", "-C", "DSR", "fetch", "--all", "--quiet"], check=True)
    target = "origin/main" if REVISION == "main" else REVISION
    subprocess.run(["git", "-C", "DSR", "checkout", "--detach", target], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "DSR"], check=True)
    os.chdir("DSR")
    print("cloned and installed")
else:
    print("not on Colab; using the existing checkout")

print("working directory:", Path.cwd())

## 1. Load code

Training, evaluation, and recovery live in the package. This cell imports them and prints
the exact revision and environment. Local modifications are recorded with source hashes;
use a clean published checkout for results intended for the report.

In [ ]:
import json
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import src.common as common
from src.common import load_config, provenance, file_hash
from src.run_m0 import baseline_caches
from src.run_m5 import run_identity, resume_status, run_m5, write_once
from src.runstore import connect

REPO = Path(common.__file__).resolve().parents[1]
PROV = provenance()
print(json.dumps(PROV, indent=2))

## 2. Settings

All experiment settings come from config.yaml. A changed protocol needs a new run label.
This notebook keeps the original class order, 0 through 29, and the original training recipe.
The only change from M4 is dense measurement of the in-training method at lambda 2.0.

In [ ]:
CFG = load_config(REPO / "config.yaml")
M5 = CFG["m5"]
SEEDS = CFG["seeds"]
D_IN = CFG["model"]["feature_dim"]
RANK = CFG["projection"]["rank"]
BUDGET = D_IN // RANK
RUN = M5["run_label"]
assert len(SEEDS) >= 3, "Reported M5 curves require at least three seeds"
assert M5["tasks"] > BUDGET, "The planned run must cross the predicted limit"
assert CFG["runtime"]["device"] == "cpu", "This notebook is CPU only"
assert M5["lambda"] == 2.0, "M5 reproduces the lambda 2.0 protocol"
print(json.dumps(CFG, indent=2))
print("Classes:", list(range(M5["tasks"])))
print("Planned adapter fits:", len(SEEDS) * M5["tasks"])
print("Input model:", CFG["model"])
print("Predicted budget:", BUDGET, "| CPU only | runtime: not yet measured")
print("Run:", RUN, "| cache:", CFG["cache"]["root"])
print("Feature tensors: about 15 MB float32 / 31 MB float64; extra dataset/dependency space needed.")

## 3. Connect to storage

Use DSR_ENDPOINT, DSR_ACCESS_KEY_ID, and DSR_SECRET_ACCESS_KEY in Colab secrets.
Existing results under this run label are downloaded and checked before recovery.
A different configuration or code revision under the same label is rejected.
Without credentials, the notebook can run, but its only copy is temporary Colab storage.
Do not cite results until persistent storage has been verified.

In [ ]:
OUT = REPO / "outputs" / RUN
OUT.mkdir(parents=True, exist_ok=True)
STORE = connect()
if STORE is None:
    print("No R2 credentials. Local results will not survive a Colab reset.")
else:
    if RUN in STORE.runs():
        STORE.get(RUN, OUT)
        print("Downloaded and hash-checked existing run:", RUN)
    print("R2 bucket:", STORE.bucket)
print("Output folder:", OUT)

## 4. Prepare the frozen model and feature caches

The shared M0 code downloads CIFAR-100 and the pinned public model if needed. It freezes
BatchNorm statistics, builds the feature cache, and checks its identity. This can take most
of the preparation time. Cache and model files stay outside the uploaded result folder.

In [ ]:
CACHE_FOLDER, CACHE_IDENTITY, HEAD, CACHES = baseline_caches(CFG, PROV)
IDENTITY = run_identity(CFG, PROV, CACHE_IDENTITY)
print("Cache:", CACHE_FOLDER)
print("Input checkpoint SHA-256:", CACHE_IDENTITY["checkpoint_sha256"])

## 5. Resume

Only verified, completed seeds are skipped. An interrupted seed is trained again from task one,
rebuilding the projector and adapters. Its existing task measurements must match the replay.
A malformed final progress record can be repaired during that replay. We do not promise
within-task or task-boundary training recovery, and do not upload model checkpoints.

Task measurements are separate immutable JSONL files. This permits incremental R2 uploads
without replacing a file that an earlier result may cite.

In [ ]:
STATUS = resume_status(OUT, IDENTITY, SEEDS, M5["tasks"])
print("Completed seeds:", STATUS["complete"])
print("Remaining seeds:", STATUS["remaining"])
if STORE is not None:
    STORE.put(OUT, RUN)

## 6. Run training and measurements

One library call trains the remaining seeds. After every deletion it records the four accuracy
readings, free rank, update size, and head changes. Progress is uploaded after every task when
R2 is connected. A failed upload stops the run so a storage problem is visible.

Task zero contains the untouched model. Forget accuracy is undefined there because no class
has been requested yet, so those entries are blank rather than an invented zero.

In [ ]:
SYNC = (lambda folder: STORE.put(folder, RUN)) if STORE is not None else None
ROWS = run_m5(CFG, HEAD, CACHES, IDENTITY, OUT, sync=SYNC)
metrics = pd.DataFrame(ROWS)
assert len(metrics) == len(SEEDS) * (M5["tasks"] + 1)
print("Verified measurement rows:", len(metrics))

## 7. Method checks before comparing performance

Check each seed separately. A zero free rank is measured, not inferred from the task number.
The numerical-zero checks only pass if at least one later task was measured.
Failure is a scientific result to inspect, not a reason to hide the accuracy curves.

In [ ]:
checks = []
for seed, part in metrics.groupby("seed"):
    hit = part.loc[part["free_rank"] == 0, "task"]
    exhausted = int(hit.min()) if len(hit) else None
    past = part[part["task"] > exhausted] if exhausted is not None else part.iloc[:0]
    expected = np.maximum(D_IN - part["task"].to_numpy() * RANK, 0)
    checks.append({
        "seed": int(seed), "measured_exhaustion": exhausted,
        "full_rank_curve_matches": bool(np.array_equal(part["free_rank"], expected)),
        "post_exhaustion_measured": bool(len(past)),
        "updates_numerically_zero": bool(len(past) and (past["update_norm"] <= M5["update_zero_atol"]).all()),
        "head_unchanged_within_tolerance": bool(len(past) and (past["weight_drift_since_exhaustion"] <= M5["weight_drift_atol"]).all()),
    })
checks_table = pd.DataFrame(checks)
print(checks_table.to_string(index=False))
COLUMNS = ["free_rank", "retain_excluding_forgotten", "forget_all", "forget_current", "retain_full"]
summary = metrics.groupby("task")[COLUMNS].agg(["mean", "std"])
print("\nSeed-level values around predicted exhaustion:")
print(metrics[metrics["task"].between(BUDGET - 1, BUDGET + 1)].to_string(index=False))
print("\nMean and standard deviation at every task:")
print(summary.round(3).to_string())

## 8. Saturation and accuracy curves

These are the M5 curves: free directions, retained-class accuracy, and forgotten-class accuracy
at lambda 2.0. Thin lines show individual seeds; bands show mean plus or minus one standard
deviation. Both forget definitions are shown because the paper does not specify which it uses.

The grey reference marks are the paper's stated values, not digitized Fig. 3 data. Our checkpoint
starts at a different accuracy. A changing retain or forget average after exhaustion can also
come from changing which classes belong to each group, even when the head is unchanged.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
plots = [("free_rank", "Directions still free"),
         ("retain_excluding_forgotten", "Retain accuracy (%)"),
         ("forget_all", "Forget accuracy (%)")]
for ax, (column, label) in zip(axes, plots):
    for seed, part in metrics.groupby("seed"):
        ax.plot(part["task"], part[column], alpha=0.25, linewidth=0.8)
    mean = summary[(column, "mean")]
    spread = summary[(column, "std")]
    ax.plot(mean.index, mean, label="mean", linewidth=2)
    ax.fill_between(mean.index, mean - spread, mean + spread, alpha=0.18, label="+/- 1 std")
    ax.axvline(BUDGET, color="red", linestyle="--", label="predicted exhaustion")
    ax.set_xlabel("Deletion requests so far")
    ax.set_ylabel(label)
    ax.grid(alpha=0.2)
axes[0].scatter([M5["paper_transition_task"]], [M5["paper_free_rank_reference"]],
                color="grey", marker="x", label="paper text reference", zorder=5)
axes[1].axhline(M5["paper_retain_reference"], color="grey", linestyle=":", label="paper text reference")
axes[2].plot(summary.index, summary[("forget_current", "mean")],
             linestyle=":", color="black", label="current class mean")
for ax in axes:
    ax.legend(fontsize=7)
fig.suptitle(f"In-training projection, lambda {M5['lambda']}, {len(SEEDS)} seeds")
fig.tight_layout()
plt.show()

### Plain-language summary

These sentences use the measured values. The paper's five-percentage-point tolerance is only
a descriptive replication reference. It is not a statistical test, and seed spread is not
uncertainty about the authors' measurements. No summary below claims that low accuracy proves
data erasure.

In [ ]:
for check in checks:
    seed = check["seed"]
    final = metrics[(metrics["seed"] == seed) & (metrics["task"] == M5["tasks"])].iloc[0]
    exhausted = check["measured_exhaustion"]
    print(f"Seed {seed}: free space reached zero at {exhausted if exhausted is not None else 'no measured task'}.")
    if check["post_exhaustion_measured"]:
        verdict = "stayed within" if check["head_unchanged_within_tolerance"] else "exceeded"
        print(f"  Later head changes {verdict} the configured tolerance.")
    print(f"  Final retain accuracy: {final['retain_excluding_forgotten']:.2f}%; "
          f"all requested classes: {final['forget_all']:.2f}%; newest class: {final['forget_current']:.2f}%.")
final_summary = summary.loc[M5["tasks"]]
print(f"Across seeds, final retain accuracy is {final_summary[('retain_excluding_forgotten', 'mean')]:.2f} "
      f"+/- {final_summary[('retain_excluding_forgotten', 'std')]:.2f} percentage points (standard deviation).")
early = metrics[metrics["task"].between(1, M5["paper_transition_task"] - 1)]
early_mean = early.groupby("task")["retain_excluding_forgotten"].mean()
within = bool(((early_mean - M5["paper_retain_reference"]).abs() <= M5["paper_tolerance_pp"]).all())
print("All pre-transition task means within the soft tolerance of the paper's retain reference:", within)
print("This is a reconstruction with a different checkpoint. These accuracy results do not establish data erasure.")

## 9. Save, reopen, and verify persistent storage

Save raw rows, seed summaries, method checks, and the figure alongside configuration and
provenance. Reopen each artifact and check row counts. If R2 is connected, download the complete
run into a fresh temporary directory and compare file hashes. Only then is persistent storage
reported as verified. Existing immutable JSON metadata is checked rather than replaced.

In [ ]:
metrics.to_csv(OUT / "metrics.csv", index=False)
summary.to_csv(OUT / "summary.csv")
checks_table.to_csv(OUT / "checks.csv", index=False)
fig.savefig(OUT / "saturation.png", dpi=160)
write_once(OUT / "metadata.json", {
    "notebook": "notebooks/05_cifar100_saturation.ipynb", "run_label": RUN,
    "protocol": IDENTITY["protocol"], "configuration": CFG, "environment": PROV,
    "cache_identity": CACHE_IDENTITY, "checks": checks,
    "resume_boundary": "completed seed; interrupted seed replays from task one",
    "decision_rule": "Full-rank capacity curve; near-zero updates and head drift after measured exhaustion, using config tolerances.",
    "limitations": ["Different checkpoint and unspecified paper training recipe.",
                    "Fixed class order and one backbone across adapter seeds.",
                    "Changing metric populations can move averages for an unchanged head.",
                    "Accuracy suppression is not proof of data erasure."],
})
assert len(pd.read_csv(OUT / "metrics.csv")) == len(ROWS)
assert len(pd.read_csv(OUT / "summary.csv", header=[0, 1], index_col=0)) == M5["tasks"] + 1
assert len(pd.read_csv(OUT / "checks.csv")) == len(SEEDS)
assert plt.imread(OUT / "saturation.png").size > 0
assert len(resume_status(OUT, IDENTITY, SEEDS, M5["tasks"])["complete"]) == len(SEEDS)
for path in sorted(OUT.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUT), path.stat().st_size, "bytes")
PERSISTENT_VERIFIED = False
if STORE is None:
    print("LOCAL ONLY. No persistent copy verified; do not cite this run yet.")
else:
    STORE.put(OUT, RUN)
    with tempfile.TemporaryDirectory() as temporary:
        downloaded = STORE.get(RUN, Path(temporary))
        for path in OUT.rglob("*"):
            if path.is_file():
                assert file_hash(path) == file_hash(downloaded / path.relative_to(OUT))
    PERSISTENT_VERIFIED = True
    print("Persistent results verified by download and hashes:", RUN)

## 10. Release the GPU

Not applicable: this notebook uses CPU only. Keep the runtime open if the preceding cell did
not verify persistent storage. M5 remains pending until this notebook has been run and its
stored results reviewed.